# KAE GPU Translator (Kaggle T4×1)

Текстовая модель Qwen2.5-7B-Instruct для перевода и refine. Не грузит
vision-модули — все параметры идут на язык, качество перевода выше.

**GPU:** Kaggle T4 (16 ГБ). Модель Qwen2.5-7B-Instruct (~14 ГБ VRAM).
Текстовая модель без vision-модулей — все параметры на язык.
Обслуживает только translate и refine.

Запускается **параллельно** с основным `runner.ipynb` (или вместо него,
если vision-задачи уже сделаны). Каждый блокнот — отдельная Kaggle-сессия
со своим туннелем.

## Секреты (Settings → Add-ons → Secrets)

| Секрет | Назначение |
|---|---|
| `KAE_MANAGER_URL` | куда слать `/runner/announce` |
| `KAE_RUNNER_TOKEN` | Bearer-токен Manager↔Runner |

In [ ]:
# 1. Репозиторий и зависимости.
!rm -rf /kaggle/working/repo
!git clone --depth=1 https://github.com/4stm4/BookAssembler.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install fastapi 'uvicorn[standard]' transformers accelerate
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. Секреты → env.
import os

_URL = '__KAE_MANAGER_URL__'
_TOKEN = '__KAE_RUNNER_TOKEN__'
if _URL.startswith('__') or _TOKEN.startswith('__'):
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    _URL = sec.get_secret('KAE_MANAGER_URL')
    _TOKEN = sec.get_secret('KAE_RUNNER_TOKEN')

os.environ['KAE_MANAGER_URL'] = _URL
os.environ['KAE_RUNNER_TOKEN'] = _TOKEN
os.environ['KAE_RUNNER_LOADERS'] = 'qwen_text'
os.environ.setdefault('KAE_RUNNER_IDLE_TIMEOUT', '900')
print('manager:', _URL)
print('loaders:', os.environ['KAE_RUNNER_LOADERS'])

In [ ]:
# 3. Диагностика GPU.
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader || echo 'нет nvidia-smi'
import torch
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else None
print('torch      :', torch.__version__)
print('cuda       :', torch.version.cuda)
print('устройство :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет CUDA')
print('capability :', f'sm_{cc[0]}{cc[1]}' if cc else '—')
print('собран под :', torch.cuda.get_arch_list() if torch.cuda.is_available() else '—')
if cc and f'sm_{cc[0]}{cc[1]}' not in torch.cuda.get_arch_list():
    print('\nВНИМАНИЕ: torch не содержит ядер под эту карту — инференс упадёт.')
    print('Перезапустите сессию, чтобы получить другую GPU.')

In [ ]:
# 4. cloudflared: публичный URL для раннера.
import itertools
import re
import subprocess

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5005'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in itertools.islice(proc.stdout, 300):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
assert url, 'cloudflared did not report a public URL'
os.environ['KAE_RUNNER_PUBLIC_URL'] = url
print('Runner will announce as:', url)

In [ ]:
# 5. Раннер (foreground). Загружает Qwen2.5-7B-Instruct (текстовый),
#    обслуживает только translate и refine.
import sys
sys.path.insert(0, '/kaggle/working/repo')
!KAE_RUNNER_HOST=0.0.0.0 python -m src.agents.runner